In [1]:
!pip install google-genai openpyxl pandas nltk gradio

# Import libraries
import pandas as pd
import numpy as np
import nltk
from google import genai
from google.colab import userdata
from google.genai import types
import string
import time
import re

# Download NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.chrf_score import sentence_chrf

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [2]:
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [4]:
file_path = 'English-Khasi Training Data 2026 (1).xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet1')

print(f"Original dataset shape: {df.shape}")
print(f"Column names: {df.columns.tolist()}")
print("\nFirst 5 rows:")
print(df.head())

df.columns = df.columns.str.strip()
df = df.rename(columns={
    'English': 'en',
    'Khasi': 'kha'
})

df = df.dropna(subset=['en', 'kha'])
print(f"\nAfter removing missing values: {df.shape}")

def clean_text(text):
    """Clean text by removing extra spaces and special characters."""
    if isinstance(text, str):
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'\n+', ' ', text)
        return text
    return text

df['en'] = df['en'].apply(clean_text)
df['kha'] = df['kha'].apply(clean_text)

df = df.drop_duplicates(subset=['en', 'kha'])
print(f"After removing duplicates: {df.shape}")

print("\nSample of cleaned data:")
for i in range(min(3, len(df))):
    print(f"English: {df.iloc[i]['en'][:100]}...")
    print(f"Khasi: {df.iloc[i]['kha'][:100]}...")
    print("-" * 50)

Original dataset shape: (26000, 2)
Column names: ['English ', 'Khasi']

First 5 rows:
                                            English   \
0  Behold , therefore I will bring strangers upon...   
1  Now when Jesus was risen early the first day o...   
2  If men strive , and hurt a woman with child , ...   
3  On the eighth day he sent the people away : an...   
4  And they of Ephraim shall be like a mighty man...   

                                               Khasi  
0  ngan wanrah ki nongshun kiba sniew ban tur ïal...  
1  Hadien ba U Jisu u la mihpat na ka jingïap dan...  
2  Lada ki rangbah kiba ïashoh ki pynmynsaw ïa ka...  
3  Ha ka sngi kaba phra u Solomon u phah noh sha ...  
4  Ki paid Israel kin long kiba khlaiñ kum ki shi...  

After removing missing values: (26000, 2)
After removing duplicates: (25995, 2)

Sample of cleaned data:
English: Behold , therefore I will bring strangers upon thee , the terrible of the nations : and they shall d...
Khasi: ngan wanrah ki nongsh

In [5]:
train_df = df.sample(n=min(25, len(df)), random_state=42)

examples_text = ""
for index, row in train_df.iterrows():
    examples_text += f"English: {row['en']}\nKhasi: {row['kha']}\n\n"

print(f"\nCreated {len(train_df)} in-context learning examples")


Created 25 in-context learning examples


In [8]:
system_instruction_text = f"""
You are an expert English to Khasi language translator.
Translate the provided English input into natural, grammatically correct Khasi.
Do not output anything else except the Khasi translation.

Here are reference translations to learn from:
{examples_text}
"""

def translate_to_khasi(english_text):
    """
    Translate English text to Khasi using Gemini API.

    Args:
        english_text (str): English text to translate

    Returns:
        str: Khasi translation
    """
    if not english_text or not english_text.strip():
        return ""

    try:
        response = client.models.generate_content(
            model='gemini-3.6-flash',  # Use the available model
            contents=english_text.strip(),
            config=types.GenerateContentConfig(
                system_instruction=system_instruction_text,
                temperature=0.3,
                max_output_tokens=1024
            )
        )
        return response.text.strip()
    except Exception as e:
        print(f"Translation error: {e}")
        return ""

test_sentence = "Good morning, how are you?"
translated_output = translate_to_khasi(test_sentence)

print(f"\nTest Translation:")
print(f"English: {test_sentence}")
print(f"Khasi: {translated_output}")


Test Translation:
English: Good morning, how are you?
Khasi: Khublei ba step, kumno phi long?


In [9]:
def evaluate_translations():
    """Evaluate translation quality on a small test set."""
    smooth = SmoothingFunction().method1

    def clean_text_for_eval(text):
        text = str(text).lower().strip()
        text = text.translate(str.maketrans('', '', string.punctuation))
        return text

    # Select test samples (not in training)
    test_df = df.drop(train_df.index).sample(n=min(10, len(df)), random_state=101).copy()

    print("\n" + "="*70)
    print("EVALUATION RESULTS")
    print("="*70)

    results = []

    for idx, (_, row) in enumerate(test_df.iterrows(), start=1):
        en_text = str(row['en']).strip()
        expected_kha = str(row['kha']).strip()

        # Translate
        predicted_kha = translate_to_khasi(en_text)

        # Calculate BLEU score
        cleaned_pred = clean_text_for_eval(predicted_kha)
        cleaned_ref = clean_text_for_eval(expected_kha)

        reference = [cleaned_ref.split()]
        candidate = cleaned_pred.split()

        bleu_score = sentence_bleu(reference, candidate, weights=(0.5, 0.5), smoothing_function=smooth) * 100

        # Calculate chrF score
        chrf_score = sentence_chrf(expected_kha.lower(), predicted_kha.lower()) * 100

        results.append({
            "Sample_ID": idx,
            "English_Input": en_text,
            "Ground_Truth_Khasi": expected_kha,
            "Model_Predicted_Khasi": predicted_kha,
            "BLEU_Score": round(bleu_score, 2),
            "chrF_Score": round(chrf_score, 2)
        })

        print(f"\nSample {idx}:")
        print(f"English: {en_text[:100]}...")
        print(f"Ground Truth: {expected_kha[:100]}...")
        print(f"Predicted: {predicted_kha[:100]}...")
        print(f"BLEU: {bleu_score:.2f}% | chrF: {chrf_score:.2f}%")
        print("-" * 50)

        # Add delay to avoid rate limits
        time.sleep(1.5)

    # Create summary DataFrame
    results_df = pd.DataFrame(results)

    summary_df = pd.DataFrame([
        {"Metric": "Total Evaluation Sentences", "Value": len(results_df)},
        {"Metric": "Average BLEU Score (%)", "Value": f"{round(results_df['BLEU_Score'].mean(), 2)}%"},
        {"Metric": "Average chrF Score (%)", "Value": f"{round(results_df['chrF_Score'].mean(), 2)}%"},
        {"Metric": "Model Version", "Value": "Gemini In-Context Learning Translator"},
        {"Metric": "Language Pair", "Value": "English → Khasi"},
        {"Metric": "Training Examples", "Value": len(train_df)}
    ])

    return results_df, summary_df

# Run evaluation
results_df, summary_df = evaluate_translations()


EVALUATION RESULTS

Sample 1:
English: If I must needs glory , I will glory of the things which concern mine infirmities ....
Ground Truth: Lada nga dei ban sngewsarong , ngan sngewsarong shaphang kiei kiei kiba pyn-i haduh katno nga long u...
Predicted: Lada nga dei ban kob , ngan kob ha kiei kiei kiba shaphang ka jingtlot jong nga ....
BLEU: 40.28% | chrF: 40.73%
--------------------------------------------------

Sample 2:
English: And what he did unto the army of Egypt , unto their horses , and to their chariots ; how he made the...
Ground Truth: Phi la ïohi kumno U Trai u la pynduh lut ïa kynhun ïapom ki nong Ijipt , lem bad ki kulai bad ki kal...
Predicted: Bad kaei kaba u la leh ha ka kynhun thma ki Barmasi , ha ki kulai jong ki , bad ha ki kali thma jong...
BLEU: 26.81% | chrF: 29.88%
--------------------------------------------------

Sample 3:
English: And I will appoint over them four kinds , saith the Lord : the sword to slay , and the dogs to tear ...
Ground Truth: Ma nga

In [10]:
output_excel_path = 'English_Khasi_Translation_Evaluation.xlsx'

with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Detailed_Evaluation', index=False)
    summary_df.to_excel(writer, sheet_name='Summary_Report', index=False)

print("\n" + "="*70)
print("FINAL EVALUATION SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)
print(f"\n Excel Report exported: '{output_excel_path}'")


FINAL EVALUATION SUMMARY
                    Metric                                 Value
Total Evaluation Sentences                                    10
    Average BLEU Score (%)                                26.11%
    Average chrF Score (%)                                36.05%
             Model Version Gemini In-Context Learning Translator
             Language Pair                       English → Khasi
         Training Examples                                    25

 Excel Report exported: 'English_Khasi_Translation_Evaluation.xlsx'


In [11]:
!pip install gradio
import gradio as gr

def gradio_translate(english_text):
    """Gradio interface function for translation."""
    if not english_text or not english_text.strip():
        return ""
    return translate_to_khasi(english_text.strip())

interface = gr.Interface(
    fn=gradio_translate,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Enter English text here to translate to Khasi...",
        label="English Input"
    ),
    outputs=gr.Textbox(
        lines=4,
        label="Khasi Translation"
    ),
    title="🇮🇳 English to Khasi Translation System",
    description="""
    ### Neural Translation Engine powered by Gemini API

    **Instructions:**
    1. Enter English text in the input box
    2. Click "Submit" or press Enter
    3. View the Khasi translation

    **Features:**
    -  In-Context Learning with 25+ training examples
    -  Real-time translation with Gemini Flash model
    -  BLEU and chrF evaluation metrics
    """,
    examples=[
        ["Good morning, my friend."],
        ["Where are you going?"],
        ["I love reading books."],
        ["What is your name?"],
        ["Thank you very much."]
    ],
    theme="huggingface"
)

# Launch the interface
interface.launch(share=True)

/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(
/usr/local/lib/python3.13/dist-packages/gradio/utils.py:621: UserWarning: Cannot load huggingface. Caught Exception: Client error '404 Not Found' for url 'https://huggingface.co/api/spaces/huggingface' (Request ID: Root=1-6a8af29b-31199d1e755339f745aad97c;da34dccb-bd78-41fd-b1ba-c2796e96ad9e)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

Sorry, we can't find the page you are looking for.
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3e2bf44e0199dee77c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
def batch_translate(input_file, output_file, translation_column='translation'):
    """
    Batch translate English text from a CSV/Excel file.

    Args:
        input_file (str): Path to input file (CSV or Excel)
        output_file (str): Path to output file
        translation_column (str): Column name for translations
    """
    if input_file.endswith('.csv'):
        df_batch = pd.read_csv(input_file)
    else:
        df_batch = pd.read_excel(input_file)

    text_columns = ['en', 'english', 'text', 'English', 'EN']
    found_col = None
    for col in text_columns:
        if col in df_batch.columns:
            found_col = col
            break

    if found_col is None:
        print("Error: Could not find English text column.")
        print(f"Available columns: {df_batch.columns.tolist()}")
        return

    print(f"\nBatch translating {len(df_batch)} sentences...")

    translations = []
    for idx, text in enumerate(df_batch[found_col].tolist()):
        if pd.isna(text) or not str(text).strip():
            translations.append("")
        else:
            translated = translate_to_khasi(str(text))
            translations.append(translated)
            print(f"Processed {idx+1}/{len(df_batch)}")
            time.sleep(0.5)  # Rate limiting

    df_batch[translation_column] = translations

    if output_file.endswith('.csv'):
        df_batch.to_csv(output_file, index=False)
    else:
        df_batch.to_excel(output_file, index=False)

    print(f"\n Batch translation complete! Saved to: {output_file}")
    return df_batch

In [13]:
def interactive_translate():
    """Run an interactive translation session."""
    print("\n" + "="*70)
    print("ENGLISH → KHASI INTERACTIVE TRANSLATOR")
    print("="*70)
    print("Type 'quit' or 'exit' to stop\n")

    while True:
        text = input("Enter English text: ").strip()

        if text.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        if not text:
            print("Please enter some text to translate.\n")
            continue

        try:
            translation = translate_to_khasi(text)
            print(f"\n📝 English: {text}")
            print(f"🇮🇳 Khasi: {translation}\n")
        except Exception as e:
            print(f"Error: {e}\n")

In [14]:
train_df.to_excel('Khasi_Training_Examples.xlsx', index=False)
print(f"\n Training examples saved to: Khasi_Training_Examples.xlsx")
print(f"   Total examples: {len(train_df)}")

print("\n" + "="*70)
print("SYSTEM READY!")
print("="*70)
print(" English to Khasi translation system is ready.")
print(" Gradio interface launched with shareable link.")
print(" Evaluation results saved to: English_Khasi_Translation_Evaluation.xlsx")
print("="*70)


 Training examples saved to: Khasi_Training_Examples.xlsx
   Total examples: 25

SYSTEM READY!
 English to Khasi translation system is ready.
 Gradio interface launched with shareable link.
 Evaluation results saved to: English_Khasi_Translation_Evaluation.xlsx


In [15]:
# ============================================================
# 7. GRADIO WEB INTERFACE
# ============================================================

!pip install gradio
import gradio as gr

def gradio_translate(english_text):
    """Gradio interface function for translation."""
    if not english_text or not english_text.strip():
        return ""
    return translate_to_khasi(english_text.strip())

# Create Gradio interface
interface = gr.Interface(
    fn=gradio_translate,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Enter English text here to translate to Khasi...",
        label="English Input"
    ),
    outputs=gr.Textbox(
        lines=4,
        label="Khasi Translation"
    ),
    title="🇮🇳 English to Khasi Translation System",
    description="""
    ### Neural Translation Engine powered by Gemini API

    **Instructions:**
    1. Enter English text in the input box
    2. Click "Submit" or press Enter
    3. View the Khasi translation

    **Features:**
    - 🔄 In-Context Learning with 25+ training examples
    - 🎯 Real-time translation with Gemini Flash model
    - 📊 BLEU and chrF evaluation metrics
    """,
    examples=[
        ["Good morning, my friend."],
        ["Where are you going?"],
        ["I love reading books."],
        ["What is your name?"],
        ["Thank you very much."]
    ],
    theme="huggingface"
)

# Launch the interface with shareable link
interface.launch(share=True)  # <-- This creates the shareable URL

/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(
/usr/local/lib/python3.13/dist-packages/gradio/utils.py:621: UserWarning: Cannot load huggingface. Caught Exception: Client error '404 Not Found' for url 'https://huggingface.co/api/spaces/huggingface' (Request ID: Root=1-6a8af461-7f5845355f9b42ef42a16537;52751c32-a6d4-4aae-957e-6664150b2753)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

Sorry, we can't find the page you are looking for.
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2886997f6816b55b35.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
